# 06 - Fuentes externas y contexto territorial por barrio

**Objetivo.** Integrar fuentes externas de CABA en una tabla territorial por barrio y comuna, sin calcular todavía distancias publicación-punto. La distancia a servicios queda para la Entrega 3; en esta etapa se construyen variables agregadas por territorio para un primer acercamiento a KPIs.

**Criterio de integración.**

- El **Censo 2022** se integra por `comuna`, porque viene agregado a ese nivel.
- Las fuentes puntuales de BA Data se agregan por `barrio_norm`, asignando cada punto al polígono oficial de barrio cuando hay coordenadas válidas.
- Si una fuente ya trae barrio pero no coordenadas confiables, se usa esa columna como llave secundaria.
- No se eliminan publicaciones por falta de contexto externo; se documenta la cobertura de la unión.

La salida principal es `features_contextuales_barrio_v1.csv`, que luego puede unirse a la base inmobiliaria para EDA y pruebas de hipótesis en el notebook 07.

In [ ]:
from pathlib import Path
import json
import re
import sys
import unicodedata
import urllib.request

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
GEOJSON_BARRIOS_LOCAL = REPO_ROOT / 'scrappers' / 'airbnb_caba' / 'barrios_caba.geojson'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('EXTERNAL_DIR:', EXTERNAL_DIR)

## Inventario de fuentes externas

Se documenta qué aporta cada fuente, cuál es su granularidad y cómo se integra en esta entrega. Las variables por distancia quedan explícitamente postergadas para la Entrega 3; ahora se generan conteos y densidades por barrio, además de variables censales por comuna.

In [ ]:
PERIODO_BADATA = 'Snapshot vigente a la descarga; verificar metadata del recurso BA Data al cierre de entrega.'

fuentes_externas = pd.DataFrame([
    {
        'id': 'F01',
        'fuente': 'Censo Nacional 2022 - poblacion, superficie y densidad por comuna',
        'organismo': 'INDEC',
        'dataset_python': 'censo_2022',
        'granularidad_original': 'comuna',
        'integracion_entrega_2': 'union por comuna',
        'variables_generadas': 'poblacion_total, superficie_km2, densidad_hab_km2',
        'aporta_a': 'oferta por habitante, densidad demografica, contexto territorial',
        'limitacion': 'no distingue barrios dentro de una misma comuna',
        'estado': 'integrada',
    },
    {
        'id': 'F02', 'fuente': 'Estaciones de subte', 'organismo': 'BA Data', 'dataset_python': 'subte_estaciones',
        'granularidad_original': 'punto', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_estaciones_subte, densidad_estaciones_subte_km2',
        'aporta_a': 'accesibilidad por transporte', 'limitacion': 'no mide distancia caminable ni frecuencia', 'estado': 'integrable por barrio',
    },
    {
        'id': 'F03', 'fuente': 'Corredores/paradas de MetroBus', 'organismo': 'BA Data', 'dataset_python': 'metrobus',
        'granularidad_original': 'punto/linea segun recurso', 'integracion_entrega_2': 'conteo/densidad por barrio si trae puntos validos',
        'variables_generadas': 'n_metrobus, densidad_metrobus_km2',
        'aporta_a': 'accesibilidad por transporte de superficie', 'limitacion': 'puede representar corredores y no paradas individuales', 'estado': 'integrable con advertencia',
    },
    {
        'id': 'F04', 'fuente': 'Estaciones de ferrocarril', 'organismo': 'BA Data', 'dataset_python': 'estaciones_ferrocarril',
        'granularidad_original': 'punto', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_estaciones_tren, densidad_estaciones_tren_km2',
        'aporta_a': 'accesibilidad metropolitana', 'limitacion': 'no mide frecuencia ni conectividad real', 'estado': 'integrable por barrio',
    },
    {
        'id': 'F05', 'fuente': 'Espacios verdes publicos', 'organismo': 'BA Data', 'dataset_python': 'espacios_verdes',
        'granularidad_original': 'punto/poligono segun recurso', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_espacios_verdes, densidad_espacios_verdes_km2',
        'aporta_a': 'entorno urbano y amenidades barriales', 'limitacion': 'conteo no mide superficie verde util ni calidad', 'estado': 'integrable por barrio',
    },
    {
        'id': 'F06', 'fuente': 'Establecimientos educativos', 'organismo': 'BA Data', 'dataset_python': 'educacion',
        'granularidad_original': 'punto', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_educacion, densidad_educacion_km2',
        'aporta_a': 'servicios urbanos y entorno residencial', 'limitacion': 'no distingue calidad, nivel ni gestion salvo analisis posterior', 'estado': 'integrable por barrio',
    },
    {
        'id': 'F07', 'fuente': 'Hospitales', 'organismo': 'BA Data', 'dataset_python': 'hospitales',
        'granularidad_original': 'punto', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_hospitales, densidad_hospitales_km2',
        'aporta_a': 'entorno de salud', 'limitacion': 'cantidad no mide capacidad ni especialidad', 'estado': 'integrable por barrio',
    },
    {
        'id': 'F08', 'fuente': 'Espacios culturales', 'organismo': 'BA Data', 'dataset_python': 'espacios_culturales',
        'granularidad_original': 'punto', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_espacios_culturales, densidad_espacios_culturales_km2',
        'aporta_a': 'atractividad barrial', 'limitacion': 'no mide asistencia ni calidad', 'estado': 'integrable por barrio',
    },
    {
        'id': 'F09', 'fuente': 'Bibliotecas publicas', 'organismo': 'BA Data', 'dataset_python': 'bibliotecas',
        'granularidad_original': 'punto', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_bibliotecas, densidad_bibliotecas_km2',
        'aporta_a': 'servicios culturales y educativos', 'limitacion': 'cobertura parcial del ecosistema cultural', 'estado': 'integrable por barrio',
    },
    {
        'id': 'F10', 'fuente': 'Gastronomia turistica', 'organismo': 'BA Data', 'dataset_python': 'gastronomia',
        'granularidad_original': 'punto', 'integracion_entrega_2': 'conteo/densidad por barrio',
        'variables_generadas': 'n_gastronomia, densidad_gastronomia_km2',
        'aporta_a': 'atractividad comercial/turistica', 'limitacion': 'registro turistico, no universo completo de locales', 'estado': 'integrable por barrio',
    },
])

display(fuentes_externas)
fuentes_externas.to_csv(PROCESSED_DIR / 'fuentes_externas_inventario_v1.csv', index=False)

## Funciones auxiliares

Estas funciones normalizan nombres de barrios, detectan columnas geográficas y asignan puntos externos a polígonos oficiales de barrio. Se usa una implementación propia de punto-en-polígono para evitar depender de GeoPandas en la ejecución docente.

In [ ]:
BBOX_CABA = {'lat': (-34.72, -34.52), 'lon': (-58.54, -58.33)}
EARTH_R_M = 6_371_000

LAT_CANDS = ['lat', 'latitud', 'latitude', 'y']
LON_CANDS = ['lon', 'long', 'lng', 'longitud', 'longitude', 'x']
COMUNA_CANDS = ['comuna', 'comunas', 'nro_comuna', 'numero_comuna']
BARRIO_CANDS = ['barrio', 'barrios', 'nombre_barrio', 'barrio_norm', 'nombre']
FECHA_HINTS = ('fecha', 'anio', 'year', 'actualiz', 'periodo')

BARRIO_ALIASES = {
    'agronomia': 'agronomia',
    'nunez': 'nunez',
    'nuñez': 'nunez',
    'monserrat': 'monserrat',
    'montserrat': 'monserrat',
    'villa gral mitre': 'villa general mitre',
    'villa general mitre': 'villa general mitre',
    'parque chacabuco': 'parque chacabuco',
    'parque chas': 'parque chas',
    'las canitas': 'palermo',
    'las cañitas': 'palermo',
}


def norm_text(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode()
    value = re.sub(r'[^a-z0-9]+', ' ', value).strip()
    value = re.sub(r'\s+', ' ', value)
    return value or pd.NA


def normalizar_barrio(value):
    value_norm = norm_text(value)
    if pd.isna(value_norm):
        return pd.NA
    return BARRIO_ALIASES.get(value_norm, value_norm)


def norm_col(c):
    c = unicodedata.normalize('NFKD', str(c)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '_', c.lower()).strip('_')


def detectar_columna(df, candidatos):
    mapa = {norm_col(c): c for c in df.columns}
    for cand in candidatos:
        if cand in mapa:
            return mapa[cand]
    return None


def a_numero(s):
    return pd.to_numeric(s.astype(str).str.replace(',', '.', regex=False), errors='coerce')


def _anillos(geom):
    if geom['type'] == 'Polygon':
        return [geom['coordinates'][0]]
    if geom['type'] == 'MultiPolygon':
        return [poly[0] for poly in geom['coordinates']]
    return []


def _en_anillo(lon, lat, anillo):
    xs = np.array([c[0] for c in anillo])
    ys = np.array([c[1] for c in anillo])
    dentro = np.zeros(len(lon), dtype=bool)
    j = len(xs) - 1
    for i in range(len(xs)):
        cruza = ((ys[i] > lat) != (ys[j] > lat)) & (lon < (xs[j] - xs[i]) * (lat - ys[i]) / (ys[j] - ys[i] + 1e-300) + xs[i])
        dentro ^= cruza
        j = i
    return dentro


def _area_km2(anillo):
    lat0 = np.radians(np.mean([c[1] for c in anillo]))
    x = np.array([np.radians(c[0]) * EARTH_R_M * np.cos(lat0) for c in anillo])
    y = np.array([np.radians(c[1]) * EARTH_R_M for c in anillo])
    return abs(0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)) / 1e6


def tabla_barrios(geojson):
    barrios = []
    for feat in geojson['features']:
        props = {norm_col(k): v for k, v in feat['properties'].items()}
        anillos = _anillos(feat['geometry'])
        nombre = props.get('nombre') or props.get('barrio') or props.get('name')
        area_m2 = props.get('area_metro')
        if area_m2 is not None and pd.notna(area_m2):
            area_km2 = float(area_m2) / 1_000_000
        else:
            area_km2 = sum(_area_km2(a) for a in anillos)
        barrios.append({
            'barrio': nombre,
            'barrio_norm': normalizar_barrio(nombre),
            'comuna': int(props.get('comuna')) if pd.notna(props.get('comuna')) else np.nan,
            'area_km2_barrio': area_km2,
            '_anillos': anillos,
        })
    return pd.DataFrame(barrios)


def asignar_barrio_por_poligono(lat, lon, barrios):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    asignado = np.full(len(lat), -1)
    validos = np.isfinite(lat) & np.isfinite(lon)
    for k, anillos in enumerate(barrios['_anillos']):
        for anillo in anillos:
            pendientes = validos & (asignado == -1)
            if pendientes.any():
                asignado[pendientes] = np.where(_en_anillo(lon[pendientes], lat[pendientes], anillo), k, -1)
    return pd.Series([barrios['barrio_norm'].iloc[k] if k >= 0 else pd.NA for k in asignado])


def cargar_geojson_barrios():
    if GEOJSON_BARRIOS_LOCAL.exists():
        with open(GEOJSON_BARRIOS_LOCAL, encoding='utf-8') as f:
            return json.load(f), 'local'
    url = 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/innovacion-transformacion-digital/barrios/barrios.geojson'
    with urllib.request.urlopen(url, timeout=60) as resp:
        return json.load(resp), 'url'


def perfilar_fuente(nombre, df):
    fila = {'dataset': nombre, 'filas': len(df), 'columnas': df.shape[1]}
    col_lat = detectar_columna(df, LAT_CANDS)
    col_lon = detectar_columna(df, LON_CANDS)
    col_barrio = detectar_columna(df, BARRIO_CANDS)
    col_comuna = detectar_columna(df, COMUNA_CANDS)
    fila.update({'col_lat': col_lat, 'col_lon': col_lon, 'col_barrio': col_barrio, 'col_comuna': col_comuna})
    if col_lat and col_lon:
        lat, lon = a_numero(df[col_lat]), a_numero(df[col_lon])
        validas = lat.notna() & lon.notna()
        en_caba = lat.between(*BBOX_CABA['lat']) & lon.between(*BBOX_CABA['lon'])
        fila['pct_coords_validas'] = round(validas.mean() * 100, 1)
        fila['pct_en_bbox_caba'] = round(en_caba.mean() * 100, 1)
    else:
        fila['pct_coords_validas'] = np.nan
        fila['pct_en_bbox_caba'] = np.nan
    fila['cols_fecha'] = ', '.join([c for c in df.columns if any(h in norm_col(c) for h in FECHA_HINTS)][:8])
    return fila

## Carga de fuentes externas y barrios oficiales

Las fuentes de BA Data se cargan desde el módulo `datasets_contextuales_caba.py`, que centraliza los enlaces. El GeoJSON de barrios se toma primero desde el archivo local del repo para reducir dependencia de internet.

In [ ]:
# Cargar datasets contextuales.
# Si algun link externo falla, el modulo devuelve None para esa fuente y el notebook sigue.
sys.path.insert(0, str(EXTERNAL_DIR))
try:
    import datasets_contextuales_caba as ctx
    datasets_ext = {k: v for k, v in ctx.datasets.items() if k != 'censo_2022'}
    print('Fuentes externas disponibles desde modulo:', sorted(datasets_ext))
except Exception as exc:
    datasets_ext = {}
    print('No se pudieron cargar las fuentes externas desde el modulo:', exc)

geo_barrios, origen_barrios = cargar_geojson_barrios()
barrios = tabla_barrios(geo_barrios)

print('GeoJSON de barrios:', origen_barrios)
print('Barrios cargados:', len(barrios))
display(barrios.drop(columns=['_anillos']).head())

perfil_fuentes = pd.DataFrame([
    perfilar_fuente(nombre, df)
    for nombre, df in datasets_ext.items()
    if df is not None
])

if not perfil_fuentes.empty:
    display(perfil_fuentes)
    perfil_fuentes.to_csv(REPORTS_DIR / 'perfil_fuentes_externas.csv', index=False)
else:
    print('No hay fuentes externas cargadas para perfilar.')

## Carga de Censo 2022 por comuna

El Censo se integra por `comuna`. No se fuerza a nivel barrio porque el cuadro usado trae población, superficie y densidad por comuna. Esta diferencia de granularidad queda documentada para no atribuir variaciones intra-comuna a una fuente agregada.

In [ ]:
ruta_censo = EXTERNAL_DIR / 'c2022_caba_est_c2_1.xlsx'

censo_raw = pd.read_excel(
    ruta_censo,
    sheet_name='Cuadro 2.1',
    skiprows=5,
    header=None,
    nrows=16,
    usecols='A:E',
    names=['codigo_comuna_indec', 'comuna_texto', 'superficie_km2', 'poblacion_total', 'densidad_hab_km2'],
)

censo_total_ciudad = censo_raw.iloc[[0]].copy()
censo_2022 = censo_raw.iloc[1:].copy()
censo_2022['comuna'] = censo_2022['comuna_texto'].astype(str).str.extract(r'(\d+)').astype(int)

for col in ['superficie_km2', 'poblacion_total', 'densidad_hab_km2']:
    censo_2022[col] = pd.to_numeric(censo_2022[col], errors='coerce')

display(censo_2022)
censo_2022.to_csv(PROCESSED_DIR / 'censo_2022_comunas_v1.csv', index=False)

## Integración de fuentes puntuales por barrio

Cada fuente se resume como cantidad de elementos por barrio y densidad por km². Si una fuente trae coordenadas válidas dentro de CABA, se asigna por polígono de barrio. Si no trae coordenadas pero sí barrio, se usa esa llave normalizada. Las fuentes que no pueden asignarse a barrio quedan documentadas en el perfil de integración.

In [ ]:
FUENTES_BARRIO = {
    'subte_estaciones': 'estaciones_subte',
    'metrobus': 'metrobus',
    'estaciones_ferrocarril': 'estaciones_tren',
    'espacios_verdes': 'espacios_verdes',
    'educacion': 'educacion',
    'hospitales': 'hospitales',
    'espacios_culturales': 'espacios_culturales',
    'bibliotecas': 'bibliotecas',
    'gastronomia': 'gastronomia',
}

features_contextuales_barrio = barrios.drop(columns=['_anillos']).copy()
perfil_integracion_barrio = []


def asignar_barrio_fuente(nombre, df):
    col_lat = detectar_columna(df, LAT_CANDS)
    col_lon = detectar_columna(df, LON_CANDS)
    col_barrio = detectar_columna(df, BARRIO_CANDS)

    if col_lat and col_lon:
        lat = a_numero(df[col_lat])
        lon = a_numero(df[col_lon])
        coords_validas = lat.between(*BBOX_CABA['lat']) & lon.between(*BBOX_CABA['lon'])
        if coords_validas.any():
            asignado = pd.Series(pd.NA, index=df.index, dtype='object')
            asignado.loc[coords_validas] = asignar_barrio_por_poligono(lat.loc[coords_validas], lon.loc[coords_validas], barrios).values
            return asignado, 'coordenadas_poligono', int(coords_validas.sum())

    if col_barrio:
        asignado = df[col_barrio].map(normalizar_barrio)
        return asignado, f'columna_barrio:{col_barrio}', int(asignado.notna().sum())

    return pd.Series(pd.NA, index=df.index, dtype='object'), 'sin_llave_barrio', 0


for nombre_dataset, prefijo in FUENTES_BARRIO.items():
    df_ext = datasets_ext.get(nombre_dataset)
    col_conteo = f'n_{prefijo}'
    col_densidad = f'densidad_{prefijo}_km2'

    features_contextuales_barrio[col_conteo] = 0

    if df_ext is None or df_ext.empty:
        features_contextuales_barrio[col_conteo] = np.nan
        features_contextuales_barrio[col_densidad] = np.nan
        perfil_integracion_barrio.append({
            'dataset': nombre_dataset,
            'variable_conteo': col_conteo,
            'filas_fuente': 0 if df_ext is None else len(df_ext),
            'filas_asignadas_barrio': 0,
            'pct_asignado_barrio': 0,
            'metodo_asignacion': 'no_cargado',
        })
        continue

    asignado, metodo, n_base_llave = asignar_barrio_fuente(nombre_dataset, df_ext)
    conteos = asignado.dropna().value_counts()

    features_contextuales_barrio[col_conteo] = (
        features_contextuales_barrio['barrio_norm']
        .map(conteos)
        .fillna(0)
        .astype(int)
    )
    features_contextuales_barrio[col_densidad] = (
        features_contextuales_barrio[col_conteo] / features_contextuales_barrio['area_km2_barrio']
    )

    perfil_integracion_barrio.append({
        'dataset': nombre_dataset,
        'variable_conteo': col_conteo,
        'filas_fuente': len(df_ext),
        'filas_con_llave_base': n_base_llave,
        'filas_asignadas_barrio': int(asignado.notna().sum()),
        'pct_asignado_barrio': round(asignado.notna().mean() * 100, 2),
        'barrios_con_elementos': int((features_contextuales_barrio[col_conteo] > 0).sum()),
        'metodo_asignacion': metodo,
    })

perfil_integracion_barrio = pd.DataFrame(perfil_integracion_barrio)

# Completar densidades de fuentes no cargadas.
for col in [c for c in features_contextuales_barrio.columns if c.startswith('n_')]:
    dens_col = f'densidad_{col[2:]}_km2'
    if dens_col not in features_contextuales_barrio.columns:
        features_contextuales_barrio[dens_col] = features_contextuales_barrio[col] / features_contextuales_barrio['area_km2_barrio']

# Agregar variables censales por comuna al nivel barrio.
features_contextuales_barrio = features_contextuales_barrio.merge(
    censo_2022[['comuna', 'poblacion_total', 'superficie_km2', 'densidad_hab_km2']],
    on='comuna',
    how='left',
    suffixes=('', '_comuna')
)

# Feature engineering territorial.
# Para esta entrega se evita distancia publicacion-punto y se trabaja con indicadores agregados por barrio.
# - n_*: cantidad de elementos en el barrio.
# - densidad_*_km2: elementos por km2 del barrio.
# - presencia_*: indica si el barrio tiene al menos un elemento de esa fuente.
# - prevalencia_*_comuna: proporción de barrios de la comuna con al menos un elemento de esa fuente.
# - tasa_*_10k_hab_comuna: disponibilidad relativa usando poblacion comunal como denominador aproximado.
# - indices_*: resumen relativo 0-100 por dimension urbana, usando ranking percentil entre barrios.

conteo_cols = [col for col in features_contextuales_barrio.columns if col.startswith('n_')]

for col in conteo_cols:
    base = col[2:]
    col_presencia = f'presencia_{base}'
    features_contextuales_barrio[col_presencia] = np.where(
        features_contextuales_barrio[col].isna(),
        np.nan,
        (features_contextuales_barrio[col] > 0).astype(int),
    )

    features_contextuales_barrio[f'prevalencia_{base}_comuna'] = (
        features_contextuales_barrio.groupby('comuna')[col_presencia].transform('mean')
    )

    if 'poblacion_total' in features_contextuales_barrio.columns:
        features_contextuales_barrio[f'tasa_{base}_10k_hab_comuna'] = (
            features_contextuales_barrio[col] / features_contextuales_barrio['poblacion_total'] * 10_000
        )


def indice_percentil(df, columnas, nombre):
    columnas = [
        col for col in columnas
        if col in df.columns and df[col].notna().any() and df[col].nunique(dropna=True) > 1
    ]
    if not columnas:
        df[nombre] = np.nan
        return df
    ranks = df[columnas].rank(pct=True).mul(100)
    df[nombre] = ranks.mean(axis=1)
    return df

features_contextuales_barrio = indice_percentil(
    features_contextuales_barrio,
    ['densidad_estaciones_subte_km2', 'densidad_metrobus_km2', 'densidad_estaciones_tren_km2'],
    'indice_accesibilidad_transporte_barrio'
)

features_contextuales_barrio = indice_percentil(
    features_contextuales_barrio,
    ['densidad_educacion_km2', 'densidad_hospitales_km2'],
    'indice_servicios_basicos_barrio'
)

features_contextuales_barrio = indice_percentil(
    features_contextuales_barrio,
    ['densidad_espacios_verdes_km2', 'densidad_espacios_culturales_km2', 'densidad_bibliotecas_km2'],
    'indice_entorno_verde_cultural_barrio'
)

features_contextuales_barrio = indice_percentil(
    features_contextuales_barrio,
    ['densidad_gastronomia_km2', 'densidad_espacios_culturales_km2', 'densidad_bibliotecas_km2'],
    'indice_atractividad_turistica_comercial_barrio'
)

features_contextuales_barrio = indice_percentil(
    features_contextuales_barrio,
    [
        'indice_accesibilidad_transporte_barrio',
        'indice_servicios_basicos_barrio',
        'indice_entorno_verde_cultural_barrio',
        'indice_atractividad_turistica_comercial_barrio',
    ],
    'indice_contexto_urbano_barrio'
)

features_contextuales_barrio.to_csv(PROCESSED_DIR / 'features_contextuales_barrio_v1.csv', index=False)
perfil_integracion_barrio.to_csv(REPORTS_DIR / 'perfil_integracion_fuentes_barrio.csv', index=False)

print('Features por barrio:', features_contextuales_barrio.shape)
display(perfil_integracion_barrio)
display(features_contextuales_barrio.head())

## Compatibilidad con la base inmobiliaria

Se carga la base consolidada más avanzada disponible y se evalúa la cobertura de `barrio_norm` y `comuna`. Esta sección no valida hipótesis: solo verifica que las llaves territoriales permitan unir contexto externo a publicaciones inmobiliarias.

In [ ]:
BASE_CONSOLIDADA_CANDIDATES = [
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_outliers_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_post_ajustes_con_flags_nulos_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_con_flags_nulos_v1.csv',
    PROCESSED_DIR / 'df_publicaciones_consolidadas_v1.csv',
]

base_path = next((path for path in BASE_CONSOLIDADA_CANDIDATES if path.exists()), None)
if base_path is None:
    raise FileNotFoundError('No se encontro base consolidada. Ejecutar primero notebooks 02-04.')

df_base = pd.read_csv(base_path, low_memory=False)
print('Base usada:', base_path.relative_to(REPO_ROOT), df_base.shape)

if 'barrio_norm' in df_base.columns:
    df_base['barrio_norm'] = df_base['barrio_norm'].map(normalizar_barrio)

compatibilidad_territorial = pd.DataFrame([
    {'metrica': 'filas_totales', 'valor': len(df_base)},
    {'metrica': 'filas_con_barrio_norm', 'valor': int(df_base['barrio_norm'].notna().sum()) if 'barrio_norm' in df_base.columns else 0},
    {'metrica': 'pct_con_barrio_norm', 'valor': round(df_base['barrio_norm'].notna().mean() * 100, 2) if 'barrio_norm' in df_base.columns else 0},
    {'metrica': 'filas_con_comuna', 'valor': int(df_base['comuna'].notna().sum()) if 'comuna' in df_base.columns else 0},
    {'metrica': 'pct_con_comuna', 'valor': round(df_base['comuna'].notna().mean() * 100, 2) if 'comuna' in df_base.columns else 0},
    {'metrica': 'barrios_en_base', 'valor': sorted(df_base['barrio_norm'].dropna().unique().tolist()) if 'barrio_norm' in df_base.columns else []},
    {'metrica': 'barrios_en_features', 'valor': sorted(features_contextuales_barrio['barrio_norm'].dropna().unique().tolist())},
])

display(compatibilidad_territorial)
compatibilidad_territorial.to_csv(REPORTS_DIR / 'compatibilidad_fuentes_externas_territorial.csv', index=False)

features_merge_cols = [c for c in features_contextuales_barrio.columns if c != 'barrio']
df_base_contexto_barrio = df_base.merge(
    features_contextuales_barrio[features_merge_cols],
    on='barrio_norm',
    how='left',
    suffixes=('', '_ctx')
)

print('Base con contexto por barrio:', df_base_contexto_barrio.shape)
print('Filas con contexto de barrio:', int(df_base_contexto_barrio['area_km2_barrio'].notna().sum()))

df_base_contexto_barrio.to_csv(PROCESSED_DIR / 'df_publicaciones_con_contexto_barrio_v1.csv', index=False)

## Consultas rápidas de prueba

Estas consultas son controles exploratorios para verificar que la integración territorial produce señales analizables. No son todavía pruebas de hipótesis: la validación estadística queda para el notebook 07.

In [ ]:
# 1. Barrios con mayor dotacion de transporte y servicios seleccionados.
cols_resumen_barrio = [
    'barrio', 'barrio_norm', 'comuna', 'area_km2_barrio',
    'n_estaciones_subte', 'n_estaciones_tren', 'n_metrobus',
    'n_espacios_verdes', 'n_educacion', 'n_hospitales', 'n_espacios_culturales',
    'densidad_estaciones_subte_km2', 'densidad_espacios_verdes_km2', 'densidad_educacion_km2',
    'presencia_estaciones_subte', 'presencia_espacios_verdes',
    'indice_accesibilidad_transporte_barrio', 'indice_servicios_basicos_barrio',
    'indice_entorno_verde_cultural_barrio', 'indice_atractividad_turistica_comercial_barrio',
    'indice_contexto_urbano_barrio',
    'poblacion_total', 'densidad_hab_km2'
]
cols_resumen_barrio = [c for c in cols_resumen_barrio if c in features_contextuales_barrio.columns]

display(
    features_contextuales_barrio[cols_resumen_barrio]
    .sort_values(['n_estaciones_subte', 'n_espacios_verdes'], ascending=False)
    .head(12)
)

# 2. Primer acercamiento a KPIs por barrio.
metricas_base = {
    'publicaciones': ('id_publicacion', 'size'),
}
if 'precio_m2_ref' in df_base_contexto_barrio.columns:
    metricas_base['precio_m2_mediana'] = ('precio_m2_ref', 'median')
if 'precio' in df_base_contexto_barrio.columns:
    metricas_base['precio_mediana'] = ('precio', 'median')

kpi_barrio_contexto = (
    df_base_contexto_barrio
    .dropna(subset=['barrio_norm'])
    .groupby(['barrio_norm', 'comuna'], dropna=False)
    .agg(**metricas_base)
    .reset_index()
    .merge(features_contextuales_barrio.drop(columns=['barrio'], errors='ignore'), on=['barrio_norm', 'comuna'], how='left')
)

if 'area_km2_barrio' in kpi_barrio_contexto.columns:
    kpi_barrio_contexto['publicaciones_por_km2_barrio'] = kpi_barrio_contexto['publicaciones'] / kpi_barrio_contexto['area_km2_barrio']

if 'poblacion_total' in kpi_barrio_contexto.columns:
    # Aproximacion comunal: el denominador es poblacion de la comuna, no del barrio.
    kpi_barrio_contexto['publicaciones_por_1000_hab_comuna'] = kpi_barrio_contexto['publicaciones'] / kpi_barrio_contexto['poblacion_total'] * 1000

kpi_barrio_contexto.to_csv(PROCESSED_DIR / 'kpi_barrio_contexto_v1.csv', index=False)

display(
    kpi_barrio_contexto
    .sort_values('publicaciones', ascending=False)
    .head(15)
)

# 3. Relaciones rapidas, solo exploratorias.
cols_contexto_prueba = [
    'densidad_estaciones_subte_km2',
    'densidad_espacios_verdes_km2',
    'densidad_educacion_km2',
    'densidad_espacios_culturales_km2',
    'indice_accesibilidad_transporte_barrio',
    'indice_servicios_basicos_barrio',
    'indice_entorno_verde_cultural_barrio',
    'indice_atractividad_turistica_comercial_barrio',
    'indice_contexto_urbano_barrio',
    'publicaciones_por_km2_barrio',
]
cols_contexto_prueba = [c for c in cols_contexto_prueba if c in kpi_barrio_contexto.columns]
cols_corr = ['precio_m2_mediana'] + cols_contexto_prueba
cols_corr = [c for c in cols_corr if c in kpi_barrio_contexto.columns]

if len(cols_corr) >= 2:
    matriz_corr_barrio = kpi_barrio_contexto[cols_corr].corr(method='spearman')
    display(matriz_corr_barrio.round(3))
    matriz_corr_barrio.to_csv(REPORTS_DIR / 'correlaciones_rapidas_contexto_barrio.csv')
else:
    print('No hay columnas suficientes para correlaciones exploratorias.')

## Cierre de fuentes externas

La integración territorial queda preparada para el notebook 07. En esta etapa se generan variables agregadas por barrio y comuna, útiles para un primer acercamiento a KPIs sin calcular distancias.

**Archivos de salida principales:**

| Archivo | Contenido |
|---|---|
| `features_contextuales_barrio_v1.csv` | Conteos, presencia, tasas relativas e indices de servicios por barrio. |
| `df_publicaciones_con_contexto_barrio_v1.csv` | Base inmobiliaria consolidada con variables contextuales por barrio. |
| `kpi_barrio_contexto_v1.csv` | Resumen de oferta, precios y contexto por barrio. |
| `perfil_integracion_fuentes_barrio.csv` | Cobertura y método de asignación de cada fuente externa. |

**Pendiente para Entrega 3:** calcular distancias publicación-punto, incorporar geocodificación donde falten coordenadas y realizar pruebas de hipótesis formales en el notebook 07.